In [ ]:
import torch
import torch.nn as nn
import numpy as np
import tqdm
import time

# -----------------------------
# Settings
# -----------------------------
NEURONS = 2
EPOCHS_ADAM = 3000
USE_LBFGS = True
EPOCHS_LBFGS = 500

N_INTERIOR = 5000          # use more than 500 if possible
N_VALIDATION_1D = 201      # 201 x 201 validation grid

LR_ADAM = 2e-3

torch.manual_seed(1234)
np.random.seed(1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64       # double precision is often helpful for PINNs

print(f"Using device: {device}")


# -----------------------------
# Exact solution and source term
# -----------------------------
def exact_u(x, y):
    """
    Exact manufactured solution:
        u = sin(pi x) sin(pi y)
    on [-1,1]^2.
    """
    return torch.sin(torch.pi * x) * torch.sin(torch.pi * y)


def exact_grad_u(x, y):
    """
    Gradient of exact solution.
    """
    ux = torch.pi * torch.cos(torch.pi * x) * torch.sin(torch.pi * y)
    uy = torch.pi * torch.sin(torch.pi * x) * torch.cos(torch.pi * y)
    return ux, uy


def source_f(x, y):
    """
    For -Delta u = f, with u = sin(pi x) sin(pi y),

        Delta u = -2 pi^2 sin(pi x) sin(pi y)

    therefore

        f = -Delta u = 2 pi^2 sin(pi x) sin(pi y).
    """
    return 2.0 * torch.pi**2 * torch.sin(torch.pi * x) * torch.sin(torch.pi * y)


# -----------------------------
# Neural network
# -----------------------------
class BaseNet(nn.Module):
    def __init__(self):
        super(BaseNet, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(2, NEURONS),
            nn.Tanh(),
            nn.Linear(NEURONS, NEURONS),
            nn.Tanh(),
            nn.Linear(NEURONS, NEURONS),
            nn.Tanh(),
            nn.Linear(NEURONS, 1)
        )

    def forward(self, x, y):
        xy = torch.cat([x, y], dim=1)
        return self.net(xy)


class PoissonDeepRitz(nn.Module):
    """
    Strongly enforces u = 0 on the boundary of [-1,1]^2 by using

        u_theta(x,y) = (1 - x^2)(1 - y^2) N_theta(x,y)

    Since (1 - x^2)(1 - y^2) = 0 whenever x = +/-1 or y = +/-1,
    the boundary condition is exactly satisfied.
    """
    def __init__(self):
        super(PoissonDeepRitz, self).__init__()
        self.base = BaseNet()

    def forward(self, x, y):
        d = (1.0 - x**2) * (1.0 - y**2)
        return d * self.base(x, y)


# -----------------------------
# Sampling
# -----------------------------
def sample_interior(n):
    """
    Uniformly sample n points from [-1,1]^2.
    """
    x = 2.0 * torch.rand(n, 1, device=device, dtype=dtype) - 1.0
    y = 2.0 * torch.rand(n, 1, device=device, dtype=dtype) - 1.0
    return x, y


# -----------------------------
# Deep Ritz energy loss
# -----------------------------
def deep_ritz_loss(model, x, y):
    """
    Energy functional for

        -Delta u = f
        u = 0 on boundary

    The weak/variational form minimizes

        J(u) = integral_Omega [ 1/2 |grad u|^2 - f u ] dx.

    Monte Carlo approximation over [-1,1]^2:

        integral ≈ area * mean(...)

    The area factor is 4 on [-1,1]^2. It is included for consistency,
    but it does not affect the optimizer minimizer.
    """
    x = x.clone().detach().requires_grad_(True)
    y = y.clone().detach().requires_grad_(True)

    u = model(x, y)

    ux = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    uy = torch.autograd.grad(
        u, y,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    f = source_f(x, y)

    energy_density = 0.5 * (ux**2 + uy**2) - f * u

    area = 4.0
    loss = area * torch.mean(energy_density)

    return loss


# -----------------------------
# Strong residual, for validation only
# -----------------------------
def poisson_residual(model, x, y):
    """
    Residual for

        -Delta u = f

    namely

        r = -u_xx - u_yy - f.

    This is used only for validation, not training.
    """
    x = x.clone().detach().requires_grad_(True)
    y = y.clone().detach().requires_grad_(True)

    u = model(x, y)

    ux = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    uy = torch.autograd.grad(
        u, y,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    uxx = torch.autograd.grad(
        ux, x,
        grad_outputs=torch.ones_like(ux),
        create_graph=True
    )[0]

    uyy = torch.autograd.grad(
        uy, y,
        grad_outputs=torch.ones_like(uy),
        create_graph=True
    )[0]

    f = source_f(x, y)

    r = -uxx - uyy - f

    return r


# -----------------------------
# Validation errors
# -----------------------------
def compute_validation_errors(model, n_1d=201, batch_size=20000):
    """
    Compute independent-grid estimates of

        L2 error
        relative L2 error
        H1 seminorm error
        relative H1 seminorm error
        PDE residual L2 norm

    using a tensor-product grid on [-1,1]^2.

    The returned L2/H1 values approximate true integral norms,
    not just RMS values.
    """
    model.eval()

    xs = torch.linspace(-1.0, 1.0, n_1d, device=device, dtype=dtype)
    ys = torch.linspace(-1.0, 1.0, n_1d, device=device, dtype=dtype)

    X, Y = torch.meshgrid(xs, ys, indexing="ij")
    x_all = X.reshape(-1, 1)
    y_all = Y.reshape(-1, 1)

    n_total = x_all.shape[0]

    sum_e2 = 0.0
    sum_u2 = 0.0

    sum_grad_e2 = 0.0
    sum_grad_u2 = 0.0

    sum_r2 = 0.0

    for i in range(0, n_total, batch_size):
        xb = x_all[i:i + batch_size]
        yb = y_all[i:i + batch_size]

        xb_req = xb.clone().detach().requires_grad_(True)
        yb_req = yb.clone().detach().requires_grad_(True)

        u_pred = model(xb_req, yb_req)
        u_true = exact_u(xb_req, yb_req)

        ux_pred = torch.autograd.grad(
            u_pred, xb_req,
            grad_outputs=torch.ones_like(u_pred),
            create_graph=True
        )[0]

        uy_pred = torch.autograd.grad(
            u_pred, yb_req,
            grad_outputs=torch.ones_like(u_pred),
            create_graph=True
        )[0]

        ux_true, uy_true = exact_grad_u(xb_req, yb_req)

        e = u_pred - u_true
        ex = ux_pred - ux_true
        ey = uy_pred - uy_true

        r = poisson_residual(model, xb, yb)

        sum_e2 += torch.sum(e**2).item()
        sum_u2 += torch.sum(u_true**2).item()

        sum_grad_e2 += torch.sum(ex**2 + ey**2).item()
        sum_grad_u2 += torch.sum(ux_true**2 + uy_true**2).item()

        sum_r2 += torch.sum(r**2).item()

    # Uniform grid approximation.
    # Integral over [-1,1]^2 is area * mean.
    area = 4.0

    mean_e2 = sum_e2 / n_total
    mean_u2 = sum_u2 / n_total

    mean_grad_e2 = sum_grad_e2 / n_total
    mean_grad_u2 = sum_grad_u2 / n_total

    mean_r2 = sum_r2 / n_total

    l2_error = np.sqrt(area * mean_e2)
    l2_exact = np.sqrt(area * mean_u2)
    rel_l2_error = l2_error / l2_exact

    h1_semi_error = np.sqrt(area * mean_grad_e2)
    h1_semi_exact = np.sqrt(area * mean_grad_u2)
    rel_h1_semi_error = h1_semi_error / h1_semi_exact

    residual_l2 = np.sqrt(area * mean_r2)

    # RMS versions are also sometimes useful for comparison with your old code.
    rms_l2_error = np.sqrt(mean_e2)
    rms_residual = np.sqrt(mean_r2)

    return {
        "L2": l2_error,
        "rel_L2": rel_l2_error,
        "H1_semi": h1_semi_error,
        "rel_H1_semi": rel_h1_semi_error,
        "residual_L2": residual_l2,
        "RMS_L2": rms_l2_error,
        "RMS_residual": rms_residual,
    }


# -----------------------------
# Train
# -----------------------------
model = PoissonDeepRitz().to(device=device, dtype=dtype)

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {num_params}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR_ADAM)

start_time = time.perf_counter()

print("\nStarting Adam training...")

with tqdm.tqdm(total=EPOCHS_ADAM, desc="Adam") as pbar:
    for epoch in range(1, EPOCHS_ADAM + 1):

        model.train()

        # Resample every epoch to reduce collocation overfitting.
        x_int, y_int = sample_interior(N_INTERIOR)

        optimizer.zero_grad()

        loss = deep_ritz_loss(model, x_int, y_int)

        loss.backward()
        optimizer.step()

        if epoch % 500 == 0 or epoch == 1:
            errs = compute_validation_errors(model, n_1d=101)
            print(
                f"Epoch {epoch:5d} | "
                f"Energy: {loss.item(): .6e} | "
                f"L2: {errs['L2']:.3e} | "
                f"rel L2: {errs['rel_L2']:.3e} | "
                f"H1 semi: {errs['H1_semi']:.3e} | "
                f"Residual L2: {errs['residual_L2']:.3e}"
            )

        pbar.update(1)


# -----------------------------
# Optional L-BFGS polishing
# -----------------------------
if USE_LBFGS:
    print("\nStarting L-BFGS polishing...")

    # L-BFGS works better with a fixed integration set.
    x_lbfgs, y_lbfgs = sample_interior(max(N_INTERIOR, 10000))

    lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=EPOCHS_LBFGS,
        max_eval=EPOCHS_LBFGS,
        tolerance_grad=1e-12,
        tolerance_change=1e-14,
        history_size=100,
        line_search_fn="strong_wolfe"
    )

    iteration = [0]

    def closure():
        lbfgs.zero_grad()

        loss = deep_ritz_loss(model, x_lbfgs, y_lbfgs)

        loss.backward()

        iteration[0] += 1

        if iteration[0] % 50 == 0 or iteration[0] == 1:
            print(f"L-BFGS iter {iteration[0]:5d} | Energy: {loss.item(): .6e}")

        return loss

    lbfgs.step(closure)


end_time = time.perf_counter()
print(f"\nTraining completed in {end_time - start_time:.3f} seconds.")


# -----------------------------
# Final validation
# -----------------------------
print("\n--- Final independent-grid verification ---")
errs = compute_validation_errors(model, n_1d=N_VALIDATION_1D)

print(f"Parameters:              {num_params}")
print(f"L2 error:                {errs['L2']:.6e}")
print(f"Relative L2 error:       {errs['rel_L2']:.6e}")
print(f"H1 seminorm error:       {errs['H1_semi']:.6e}")
print(f"Relative H1 semi error:  {errs['rel_H1_semi']:.6e}")
print(f"Residual L2 norm:        {errs['residual_L2']:.6e}")
print(f"RMS L2 error:            {errs['RMS_L2']:.6e}")
print(f"RMS residual:            {errs['RMS_residual']:.6e}")

|Parameters|Accuracy|
|:---:|:---:|
| 57 | 3.083089e-02 |
| 177 | 3.095326e-02 |
| 609| 3.176074e-02 |
|2241 |3.479839e-02 |

In [ ]:
def run_case(neurons, n_interior, seed=1234):
    global NEURONS, N_INTERIOR

    NEURONS = neurons
    N_INTERIOR = n_interior

    torch.manual_seed(seed)
    np.random.seed(seed)

    model = PoissonDeepRitz().to(device=device, dtype=dtype)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR_ADAM)

    for epoch in range(1, EPOCHS_ADAM + 1):
        model.train()
        x_int, y_int = sample_interior(N_INTERIOR)

        optimizer.zero_grad()
        loss = deep_ritz_loss(model, x_int, y_int)
        loss.backward()
        optimizer.step()

    if USE_LBFGS:
        x_lbfgs, y_lbfgs = sample_interior(max(N_INTERIOR, 10000))

        lbfgs = torch.optim.LBFGS(
            model.parameters(),
            lr=1.0,
            max_iter=EPOCHS_LBFGS,
            max_eval=EPOCHS_LBFGS,
            tolerance_grad=1e-12,
            tolerance_change=1e-14,
            history_size=100,
            line_search_fn="strong_wolfe"
        )

        def closure():
            lbfgs.zero_grad()
            loss = deep_ritz_loss(model, x_lbfgs, y_lbfgs)
            loss.backward()
            return loss

        lbfgs.step(closure)

    errs = compute_validation_errors(model, n_1d=N_VALIDATION_1D)

    num_params = sum(p.numel() for p in model.parameters())

    return {
        "neurons": neurons,
        "N_INTERIOR": n_interior,
        "parameters": num_params,
        "L2": errs["L2"],
        "rel_L2": errs["rel_L2"],
        "H1_semi": errs["H1_semi"],
        "rel_H1_semi": errs["rel_H1_semi"],
        "residual_L2": errs["residual_L2"],
    }

In [ ]:
results = []

NEURONS_LIST = [8, 16, 32, 64, 128]

for neurons in NEURONS_LIST:
    print(f"\nRunning NEURONS = {neurons}")
    result = run_case(
        neurons=neurons,
        n_interior=20000,
        seed=1234
    )
    results.append(result)
    print(result)


print("\nConvergence with respect to network size")
print("neurons, params, L2, rel_L2, H1_semi, residual_L2")

for r in results:
    print(
        r["neurons"],
        r["parameters"],
        f"{r['L2']:.3e}",
        f"{r['rel_L2']:.3e}",
        f"{r['H1_semi']:.3e}",
        f"{r['residual_L2']:.3e}"
    )



In [ ]:
results = []

N_INTERIOR_LIST = [100000,200000]

for n_interior in N_INTERIOR_LIST:
    print(f"\nRunning N_INTERIOR = {n_interior}")
    result = run_case(
        neurons=64,
        n_interior=n_interior,
        seed=1234
    )
    result["h_eff"] = 2.0 / np.sqrt(n_interior)
    results.append(result)
    print(result)

print("\nConvergence with respect to N_INTERIOR")
print("N, h_eff, L2, rate_L2, H1, rate_H1")

for i, r in enumerate(results):
    if i == 0:
        print(
            r["N_INTERIOR"],
            f"{r['h_eff']:.3e}",
            f"{r['L2']:.3e}",
            "---",
            f"{r['H1_semi']:.3e}",
            "---"
        )
    else:
        r_prev = results[i - 1]

        rate_l2 = np.log(r_prev["L2"] / r["L2"]) / np.log(r_prev["h_eff"] / r["h_eff"])
        rate_h1 = np.log(r_prev["H1_semi"] / r["H1_semi"]) / np.log(r_prev["h_eff"] / r["h_eff"])

        print(
            r["N_INTERIOR"],
            f"{r['h_eff']:.3e}",
            f"{r['L2']:.3e}",
            f"{rate_l2:.2f}",
            f"{r['H1_semi']:.3e}",
            f"{rate_h1:.2f}"
        )

In [1]:
import torch
import torch.nn as nn
import numpy as np
import tqdm
import time


# ============================================================
# Global settings
# ============================================================

torch.manual_seed(1234)
np.random.seed(1234)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float64

print(f"Using device: {device}")

# Training settings
EPOCHS_ADAM = 3000
USE_LBFGS = True
EPOCHS_LBFGS = 500
LR_ADAM = 2.0e-3

# Validation settings
N_VALIDATION_1D = 301
VALIDATION_BATCH_SIZE = 20000

# Sampling settings
USE_SOBOL = True

# Important sign convention.
#
# The source term you provided appears to be:
#
#     f_given = Delta_Gamma u_exact
#
# because its flat leading term is -2*pi^2*sin(pi*x)*sin(pi*y).
#
# The Deep Ritz energy below solves:
#
#     -Delta_Gamma u = f
#
# Therefore, we use:
#
#     f = -f_given
#
SOURCE_GIVEN_IS_DELTA_GAMMA_U = True


# ============================================================
# Exact solution
# ============================================================

def exact_u(x, y):
    """
    Exact manufactured solution:
        u(x,y) = sin(pi x) sin(pi y)
    """
    return torch.sin(torch.pi * x) * torch.sin(torch.pi * y)


def exact_grad_u(x, y):
    """
    Parameter-space gradient of exact solution.
    """
    ux = torch.pi * torch.cos(torch.pi * x) * torch.sin(torch.pi * y)
    uy = torch.pi * torch.sin(torch.pi * x) * torch.cos(torch.pi * y)
    return ux, uy


# ============================================================
# Surface geometry
# ============================================================

def surface_height(x, y):
    """
    Surface graph:
        z = phi(x,y) = 1/pi * sin(2*pi*(x^2+y^2))
    """
    R = x**2 + y**2
    return (1.0 / torch.pi) * torch.sin(2.0 * torch.pi * R)


def surface_geometry(x, y):
    """
    Geometry for the graph surface

        Gamma = {(x,y,z): z = phi(x,y)}

    with

        phi(x,y) = 1/pi * sin(2*pi*(x^2+y^2)).

    Returns:
        sqrt_g : surface area factor sqrt(det(g))
        ginv11 : inverse metric component g^{11}
        ginv12 : inverse metric component g^{12}
        ginv22 : inverse metric component g^{22}
        phi_x  : derivative of surface height wrt x
        phi_y  : derivative of surface height wrt y
    """
    R = x**2 + y**2

    c = torch.cos(2.0 * torch.pi * R)

    # phi_x = d/dx [1/pi sin(2*pi R)]
    #       = 1/pi * cos(2*pi R) * 2*pi * 2x
    #       = 4x cos(2*pi R)
    phi_x = 4.0 * x * c
    phi_y = 4.0 * y * c

    D = 1.0 + phi_x**2 + phi_y**2
    sqrt_g = torch.sqrt(D)

    # Inverse metric for graph surface:
    #
    # g^{-1} = I - grad(phi) grad(phi)^T / (1 + |grad(phi)|^2)
    ginv11 = 1.0 - phi_x**2 / D
    ginv22 = 1.0 - phi_y**2 / D
    ginv12 = -phi_x * phi_y / D

    return sqrt_g, ginv11, ginv12, ginv22, phi_x, phi_y


# ============================================================
# Source term
# ============================================================

def source_laplace_given(x, y):
    """
    This is your provided source term, translated to PyTorch.

    It appears to represent

        f_given = Delta_Gamma u_exact.

    The Deep Ritz formulation below solves

        -Delta_Gamma u = f,

    so the training source is usually

        f = -source_laplace_given.
    """
    R = x**2 + y**2

    c = torch.cos(2.0 * torch.pi * R)
    s = torch.sin(2.0 * torch.pi * R)

    D = 1.0 + 16.0 * R * c**2

    f_sin = torch.sin(torch.pi * x) * torch.sin(torch.pi * y)
    f_cos = torch.cos(torch.pi * x) * torch.cos(torch.pi * y)

    term1 = (
        -torch.pi**2 * (2.0 + 16.0 * R * c**2) / D
        * f_sin
    )

    term2 = (
        -32.0 * torch.pi**2 * x * y * c**2 / D
        * f_cos
    )

    bracket = (
        x * torch.cos(torch.pi * x) * torch.sin(torch.pi * y)
        + y * torch.sin(torch.pi * x) * torch.cos(torch.pi * y)
    )

    term3 = (
        -32.0 * torch.pi * c
        * (c * (1.0 + 8.0 * R * c**2) - 2.0 * torch.pi * R * s)
        / D**2
        * bracket
    )

    return term1 + term2 + term3


def source_for_minus_laplace_equation(x, y):
    """
    Source f for the equation

        -Delta_Gamma u = f.

    If source_laplace_given = Delta_Gamma u_exact,
    then f = -source_laplace_given.
    """
    f_given = source_laplace_given(x, y)

    if SOURCE_GIVEN_IS_DELTA_GAMMA_U:
        return -f_given
    else:
        return f_given


# ============================================================
# Neural network
# ============================================================

class BaseNet(nn.Module):
    def __init__(self, neurons):
        super(BaseNet, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(2, neurons),
            nn.Tanh(),
            nn.Linear(neurons, neurons),
            nn.Tanh(),
            nn.Linear(neurons, neurons),
            nn.Tanh(),
            nn.Linear(neurons, 1)
        )

    def forward(self, x, y):
        xy = torch.cat([x, y], dim=1)
        return self.net(xy)


class SurfacePoissonDeepRitz(nn.Module):
    """
    Strongly enforces homogeneous Dirichlet boundary conditions on [-1,1]^2:

        u_theta(x,y) = (1 - x^2)(1 - y^2) N_theta(x,y)

    Since the exact solution sin(pi x) sin(pi y) vanishes on x=±1 or y=±1,
    this is appropriate.
    """
    def __init__(self, neurons):
        super(SurfacePoissonDeepRitz, self).__init__()
        self.base = BaseNet(neurons)

    def forward(self, x, y):
        boundary_factor = (1.0 - x**2) * (1.0 - y**2)
        return boundary_factor * self.base(x, y)


# ============================================================
# Sampling
# ============================================================

def sample_interior(n, sobol_engine=None):
    """
    Sample n points from the parameter domain [-1,1]^2.

    If sobol_engine is provided, use Sobol points.
    Otherwise, use random uniform points.
    """
    if sobol_engine is not None:
        pts = sobol_engine.draw(n).to(device=device, dtype=dtype)
        pts = 2.0 * pts - 1.0
        x = pts[:, 0:1]
        y = pts[:, 1:2]
    else:
        x = 2.0 * torch.rand(n, 1, device=device, dtype=dtype) - 1.0
        y = 2.0 * torch.rand(n, 1, device=device, dtype=dtype) - 1.0

    return x, y


# ============================================================
# Surface Deep Ritz loss
# ============================================================

def surface_deep_ritz_loss(model, x, y):
    """
    Deep Ritz energy for

        -Delta_Gamma u = f

    on the graph surface Gamma.

    The continuous energy is

        J(u) = integral_Gamma [ 1/2 |grad_Gamma u|^2 - f u ] dS.

    In parameter coordinates:

        J(u) =
            integral_{[-1,1]^2}
            [ 1/2 grad(u)^T g^{-1} grad(u) - f u ] sqrt(g) dx dy.
    """
    x = x.clone().detach().requires_grad_(True)
    y = y.clone().detach().requires_grad_(True)

    u = model(x, y)

    ux = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    uy = torch.autograd.grad(
        u, y,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    sqrt_g, ginv11, ginv12, ginv22, _, _ = surface_geometry(x, y)

    grad_gamma_norm_sq = (
        ginv11 * ux**2
        + 2.0 * ginv12 * ux * uy
        + ginv22 * uy**2
    )

    f = source_for_minus_laplace_equation(x, y)

    energy_density = 0.5 * grad_gamma_norm_sq - f * u

    # Parameter domain area is 4.
    area_param = 4.0

    return area_param * torch.mean(sqrt_g * energy_density)


# ============================================================
# Laplace-Beltrami operator and residual
# ============================================================

def laplace_beltrami(model, x, y):
    """
    Computes Delta_Gamma u for a graph surface.

    Formula:

        Delta_Gamma u =
            1/sqrt(g) * d_i( sqrt(g) g^{ij} d_j u )

    where i,j run over x,y.
    """
    x = x.clone().detach().requires_grad_(True)
    y = y.clone().detach().requires_grad_(True)

    u = model(x, y)

    ux = torch.autograd.grad(
        u, x,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    uy = torch.autograd.grad(
        u, y,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]

    sqrt_g, ginv11, ginv12, ginv22, _, _ = surface_geometry(x, y)

    A_x = sqrt_g * (ginv11 * ux + ginv12 * uy)
    A_y = sqrt_g * (ginv12 * ux + ginv22 * uy)

    dA_x_dx = torch.autograd.grad(
        A_x, x,
        grad_outputs=torch.ones_like(A_x),
        create_graph=True
    )[0]

    dA_y_dy = torch.autograd.grad(
        A_y, y,
        grad_outputs=torch.ones_like(A_y),
        create_graph=True
    )[0]

    delta_gamma_u = (dA_x_dx + dA_y_dy) / sqrt_g

    return delta_gamma_u


def surface_residual(model, x, y):
    """
    Strong residual for the equation

        -Delta_Gamma u = f.

    Returns

        r = -Delta_Gamma u_theta - f.
    """
    delta_gamma_u = laplace_beltrami(model, x, y)
    f = source_for_minus_laplace_equation(x, y)

    return -delta_gamma_u - f


# ============================================================
# Surface validation errors
# ============================================================

def compute_surface_validation_errors(model, n_1d=301, batch_size=20000):
    """
    Computes surface-weighted validation errors:

        ||u - u_theta||_{L2(Gamma)}
        relative L2(Gamma)
        |u - u_theta|_{H1(Gamma)}
        relative H1 seminorm
        ||-Delta_Gamma u_theta - f||_{L2(Gamma)}

    The integrals are approximated over the parameter square [-1,1]^2
    using the surface measure:

        dS = sqrt(g) dx dy.
    """
    model.eval()

    xs = torch.linspace(-1.0, 1.0, n_1d, device=device, dtype=dtype)
    ys = torch.linspace(-1.0, 1.0, n_1d, device=device, dtype=dtype)

    X, Y = torch.meshgrid(xs, ys, indexing="ij")

    x_all = X.reshape(-1, 1)
    y_all = Y.reshape(-1, 1)

    n_total = x_all.shape[0]

    sum_e2 = 0.0
    sum_u2 = 0.0

    sum_grad_e2 = 0.0
    sum_grad_u2 = 0.0

    sum_r2 = 0.0
    sum_sqrt_g = 0.0

    for i in range(0, n_total, batch_size):
        xb = x_all[i:i + batch_size]
        yb = y_all[i:i + batch_size]

        xb_req = xb.clone().detach().requires_grad_(True)
        yb_req = yb.clone().detach().requires_grad_(True)

        u_pred = model(xb_req, yb_req)
        u_true = exact_u(xb_req, yb_req)

        ux_pred = torch.autograd.grad(
            u_pred, xb_req,
            grad_outputs=torch.ones_like(u_pred),
            create_graph=True
        )[0]

        uy_pred = torch.autograd.grad(
            u_pred, yb_req,
            grad_outputs=torch.ones_like(u_pred),
            create_graph=True
        )[0]

        ux_true, uy_true = exact_grad_u(xb_req, yb_req)

        e = u_pred - u_true
        ex = ux_pred - ux_true
        ey = uy_pred - uy_true

        sqrt_g, ginv11, ginv12, ginv22, _, _ = surface_geometry(xb_req, yb_req)

        grad_e_gamma_sq = (
            ginv11 * ex**2
            + 2.0 * ginv12 * ex * ey
            + ginv22 * ey**2
        )

        grad_u_gamma_sq = (
            ginv11 * ux_true**2
            + 2.0 * ginv12 * ux_true * uy_true
            + ginv22 * uy_true**2
        )

        r = surface_residual(model, xb, yb)

        sum_e2 += torch.sum(sqrt_g * e**2).item()
        sum_u2 += torch.sum(sqrt_g * u_true**2).item()

        sum_grad_e2 += torch.sum(sqrt_g * grad_e_gamma_sq).item()
        sum_grad_u2 += torch.sum(sqrt_g * grad_u_gamma_sq).item()

        sum_r2 += torch.sum(sqrt_g * r**2).item()
        sum_sqrt_g += torch.sum(sqrt_g).item()

    area_param = 4.0

    mean_e2 = sum_e2 / n_total
    mean_u2 = sum_u2 / n_total

    mean_grad_e2 = sum_grad_e2 / n_total
    mean_grad_u2 = sum_grad_u2 / n_total

    mean_r2 = sum_r2 / n_total
    mean_sqrt_g = sum_sqrt_g / n_total

    surface_area = area_param * mean_sqrt_g

    l2_error = np.sqrt(area_param * mean_e2)
    l2_exact = np.sqrt(area_param * mean_u2)
    rel_l2_error = l2_error / l2_exact

    h1_semi_error = np.sqrt(area_param * mean_grad_e2)
    h1_semi_exact = np.sqrt(area_param * mean_grad_u2)
    rel_h1_semi_error = h1_semi_error / h1_semi_exact

    residual_l2 = np.sqrt(area_param * mean_r2)

    rms_l2_error = np.sqrt((area_param * mean_e2) / surface_area)
    rms_residual = np.sqrt((area_param * mean_r2) / surface_area)

    return {
        "surface_area": surface_area,
        "L2": l2_error,
        "rel_L2": rel_l2_error,
        "H1_semi": h1_semi_error,
        "rel_H1_semi": rel_h1_semi_error,
        "residual_L2": residual_l2,
        "RMS_L2": rms_l2_error,
        "RMS_residual": rms_residual,
    }


# ============================================================
# Single training run
# ============================================================

def run_case(
    neurons,
    n_interior,
    seed=1234,
    epochs_adam=EPOCHS_ADAM,
    use_lbfgs=USE_LBFGS,
    epochs_lbfgs=EPOCHS_LBFGS,
    n_validation_1d=N_VALIDATION_1D,
    verbose=True
):
    """
    Trains one surface Deep Ritz PINN and returns validation errors.
    """

    torch.manual_seed(seed)
    np.random.seed(seed)

    model = SurfacePoissonDeepRitz(neurons).to(device=device, dtype=dtype)

    num_params = sum(p.numel() for p in model.parameters())

    if verbose:
        print("\n" + "=" * 70)
        print(f"Running case: NEURONS = {neurons}, N_INTERIOR = {n_interior}")
        print(f"Parameters: {num_params}")
        print("=" * 70)

    # Sobol engine for this run.
    if USE_SOBOL:
        sobol_engine = torch.quasirandom.SobolEngine(
            dimension=2,
            scramble=True,
            seed=seed
        )
    else:
        sobol_engine = None

    optimizer = torch.optim.Adam(model.parameters(), lr=LR_ADAM)

    start_time = time.perf_counter()

    # -------------------------
    # Adam training
    # -------------------------
    if verbose:
        print("Starting Adam training...")

    pbar = tqdm.tqdm(total=epochs_adam, desc="Adam", disable=not verbose)

    for epoch in range(1, epochs_adam + 1):
        model.train()

        # Resample integration points each epoch.
        x_int, y_int = sample_interior(n_interior, sobol_engine=sobol_engine)

        optimizer.zero_grad()
        loss = surface_deep_ritz_loss(model, x_int, y_int)
        loss.backward()
        optimizer.step()

        if verbose and (epoch == 1 or epoch % 500 == 0):
            errs = compute_surface_validation_errors(
                model,
                n_1d=101,
                batch_size=VALIDATION_BATCH_SIZE
            )

            print(
                f"Epoch {epoch:5d} | "
                f"Energy: {loss.item(): .6e} | "
                f"L2: {errs['L2']:.3e} | "
                f"rel L2: {errs['rel_L2']:.3e} | "
                f"H1 semi: {errs['H1_semi']:.3e} | "
                f"Residual L2: {errs['residual_L2']:.3e}"
            )

        pbar.update(1)

    pbar.close()

    # -------------------------
    # L-BFGS polishing
    # -------------------------
    if use_lbfgs:
        if verbose:
            print("\nStarting L-BFGS polishing...")

        # L-BFGS uses a fixed integration set.
        if USE_SOBOL:
            lbfgs_engine = torch.quasirandom.SobolEngine(
                dimension=2,
                scramble=True,
                seed=seed + 999
            )
        else:
            lbfgs_engine = None

        x_lbfgs, y_lbfgs = sample_interior(n_interior, sobol_engine=lbfgs_engine)

        lbfgs = torch.optim.LBFGS(
            model.parameters(),
            lr=1.0,
            max_iter=epochs_lbfgs,
            max_eval=epochs_lbfgs,
            tolerance_grad=1e-12,
            tolerance_change=1e-14,
            history_size=100,
            line_search_fn="strong_wolfe"
        )

        iteration = [0]

        def closure():
            lbfgs.zero_grad()
            loss = surface_deep_ritz_loss(model, x_lbfgs, y_lbfgs)
            loss.backward()

            iteration[0] += 1

            if verbose and (iteration[0] == 1 or iteration[0] % 50 == 0):
                print(f"L-BFGS iter {iteration[0]:5d} | Energy: {loss.item(): .6e}")

            return loss

        lbfgs.step(closure)

    end_time = time.perf_counter()

    # -------------------------
    # Final validation
    # -------------------------
    errs = compute_surface_validation_errors(
        model,
        n_1d=n_validation_1d,
        batch_size=VALIDATION_BATCH_SIZE
    )

    h_eff_param = 2.0 / np.sqrt(n_interior)
    h_eff_surface = np.sqrt(errs["surface_area"] / n_interior)

    result = {
        "neurons": neurons,
        "N_INTERIOR": n_interior,
        "parameters": num_params,
        "surface_area": errs["surface_area"],
        "h_eff_param": h_eff_param,
        "h_eff_surface": h_eff_surface,
        "L2": errs["L2"],
        "rel_L2": errs["rel_L2"],
        "H1_semi": errs["H1_semi"],
        "rel_H1_semi": errs["rel_H1_semi"],
        "residual_L2": errs["residual_L2"],
        "RMS_L2": errs["RMS_L2"],
        "RMS_residual": errs["RMS_residual"],
        "training_time": end_time - start_time,
    }

    if verbose:
        print("\n--- Final independent surface-grid verification ---")
        print(f"Parameters:              {num_params}")
        print(f"Surface area estimate:   {result['surface_area']:.6e}")
        print(f"h_eff_param:             {result['h_eff_param']:.6e}")
        print(f"h_eff_surface:           {result['h_eff_surface']:.6e}")
        print(f"L2 error:                {result['L2']:.6e}")
        print(f"Relative L2 error:       {result['rel_L2']:.6e}")
        print(f"H1 seminorm error:       {result['H1_semi']:.6e}")
        print(f"Relative H1 semi error:  {result['rel_H1_semi']:.6e}")
        print(f"Residual L2 norm:        {result['residual_L2']:.6e}")
        print(f"RMS L2 error:            {result['RMS_L2']:.6e}")
        print(f"RMS residual:            {result['RMS_residual']:.6e}")
        print(f"Training time:           {result['training_time']:.3f} seconds")

    return result


# ============================================================
# Rate computation
# ============================================================

def print_convergence_table(results, h_key="h_eff_surface"):
    """
    Print convergence table using either:

        h_key = "h_eff_surface"

    or

        h_key = "h_eff_param".
    """
    print("\n" + "=" * 90)
    print(f"Convergence table using {h_key}")
    print("=" * 90)

    print(
        "N_INTERIOR, h_eff, L2, rate_L2, H1_semi, rate_H1, residual_L2, rate_res"
    )

    for i, r in enumerate(results):
        if i == 0:
            print(
                f"{r['N_INTERIOR']:8d} "
                f"{r[h_key]:.6e} "
                f"{r['L2']:.6e} "
                f"{'---':>8} "
                f"{r['H1_semi']:.6e} "
                f"{'---':>8} "
                f"{r['residual_L2']:.6e} "
                f"{'---':>8}"
            )
        else:
            r_prev = results[i - 1]

            h1 = r_prev[h_key]
            h2 = r[h_key]

            rate_l2 = np.log(r_prev["L2"] / r["L2"]) / np.log(h1 / h2)
            rate_h1 = np.log(r_prev["H1_semi"] / r["H1_semi"]) / np.log(h1 / h2)
            rate_res = np.log(r_prev["residual_L2"] / r["residual_L2"]) / np.log(h1 / h2)

            print(
                f"{r['N_INTERIOR']:8d} "
                f"{r[h_key]:.6e} "
                f"{r['L2']:.6e} "
                f"{rate_l2:8.3f} "
                f"{r['H1_semi']:.6e} "
                f"{rate_h1:8.3f} "
                f"{r['residual_L2']:.6e} "
                f"{rate_res:8.3f}"
            )


# ============================================================
# Main experiment
# ============================================================

if __name__ == "__main__":

    # For first testing, keep this list small.
    # For a serious convergence test, increase to:
    #
    # N_INTERIOR_LIST = [20000, 50000, 100000, 200000]
    #
    # or larger if your GPU can handle it.
    N_INTERIOR_LIST = [
        200000
    ]

    NEURONS = 64

    results = []

    for n_int in N_INTERIOR_LIST:
        result = run_case(
            neurons=NEURONS,
            n_interior=n_int,
            seed=1234,
            epochs_adam=EPOCHS_ADAM,
            use_lbfgs=USE_LBFGS,
            epochs_lbfgs=EPOCHS_LBFGS,
            n_validation_1d=N_VALIDATION_1D,
            verbose=True
        )

        results.append(result)

        print("\nResult dictionary:")
        print(result)

    print_convergence_table(results, h_key="h_eff_surface")

    print("\nFor comparison, using parameter-domain h_eff:")
    print_convergence_table(results, h_key="h_eff_param")

Using device: cuda

Running case: NEURONS = 64, N_INTERIOR = 200000
Parameters: 8577
Starting Adam training...


Adam:   0%|          | 1/3000 [00:00<45:23,  1.10it/s]

Epoch     1 | Energy: -6.575192e-02 | L2: 1.454e+00 | rel L2: 9.920e-01 | H1 semi: 5.006e+00 | Residual L2: 2.855e+01


Adam:  17%|█▋        | 500/3000 [04:53<32:23,  1.29it/s]

Epoch   500 | Energy: -1.279221e+01 | L2: 5.353e-02 | rel L2: 3.653e-02 | H1 semi: 1.777e-01 | Residual L2: 1.946e+00


...

Adam:  83%|████████▎ | 2500/3000 [23:26<04:49,  1.72it/s]

Epoch  2500 | Energy: -1.280490e+01 | L2: 4.020e-02 | rel L2: 2.743e-02 | H1 semi: 7.332e-02 | Residual L2: 7.371e-01


Adam: 100%|██████████| 3000/3000 [26:56<00:00,  1.86it/s]

Epoch  3000 | Energy: -1.279749e+01 | L2: 7.432e-03 | rel L2: 5.071e-03 | H1 semi: 3.411e-02 | Residual L2: 5.303e-01

Starting L-BFGS polishing...


L-BFGS iter     1 | Energy: -1.280185e+01
L-BFGS iter    50 | Energy: -1.280217e+01
L-BFGS iter   100 | Energy: -1.280229e+01
L-BFGS iter   150 | Energy: -1.280232e+01
L-BFGS iter   200 | Energy: -1.280234e+01
L-BFGS iter   250 | Energy: -1.280235e+01
L-BFGS iter   300 | Energy: -1.280236e+01
L-BFGS iter   350 | Energy: -1.280236e+01
L-BFGS iter   400 | Energy: -1.280236e+01
L-BFGS iter   450 | Energy: -1.280236e+01
L-BFGS iter   500 | Energy: -1.280236e+01

--- Final independent surface-grid verification ---
Parameters:              8577
Surface area estimate:   9.186623e+00
h_eff_param:             4.472136e-03
h_eff_surface:           6.777397e-03
L2 error:                5.921209e-04
Relative L2 error:       4.013814e-04
H1 seminorm error:       5.166152e-03
Relative H1 semi error:  1.021857e-03
Residual L2 norm:        9.366943e-02
RMS L2 error:            1.953586e-04
RMS residual:            3.090437e-02
Training time:           1790.632 seconds

Result dictionary:
{'neurons': 6